## Imports

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

from scipy import stats
from statsmodels.stats.multitest import fdrcorrection

import shap

import pickle
import sys

sys.path.append("../")
from prediction.loss import *
from prediction.data_structures import *
from prediction.model import *

## Data

In [ ]:
# single cell data
PATH_TO_DATA = '/ems/elsc-labs/habib-n/yuval.rom/500/DeepDynamics/prediction/data/500.h5ad'
data = sc.read_h5ad(PATH_TO_DATA)
data

In [ ]:
# Filtering to only the most confident cell states (68)
cell_corr_data = data.uns['celmod']['test.corrs']
cell_state_filter = cell_corr_data.index[(cell_corr_data['adj.pval'] < 0.005) & (cell_corr_data['corr'] > 0)]
cell_state_filter.shape

In [ ]:
shared_bulk_data_mask = pd.read_csv('/ems/elsc-labs/habib-n/yuval.rom/500/DeepDynamics/prediction/data/shared_bulk_data_0.005.csv', index_col=0)

In [ ]:
# This is the bulk samples that do not appears on the single cell data, with filtered cell types
unshared_bulk_data = data.uns['celmod']['avg.predicted.prop']['validation'][cell_state_filter]
unshared_bulk_data.shape

In [ ]:
all_bulk_data = pd.concat((shared_bulk_data_mask, unshared_bulk_data))
all_bulk_data.shape

## Load model

In [ ]:
model_thin = ProbModel()
model_thin.load_state_dict(torch.load('/ems/elsc-labs/habib-n/yuval.rom/500/DeepDynamics/prediction/weights/model.pth'))

## Explainability

In [ ]:
shap.initjs()
def model_wrapper(x):
    x_tensor = torch.tensor(x).float()
    y_pred = model_thin(x_tensor)
    return y_pred.detach().numpy()


In [ ]:
titles = ['prAD', 'ABA', 'Pseudotime']
for i in range(len(titles)):
    plt.figure(figsize=(10, 8))  # Adjust the figure size as needed

    # Create a custom color map
    from matplotlib.colors import LinearSegmentedColormap
    custom_cmap = LinearSegmentedColormap.from_list("custom", ["#4db6ac", "#FFFFFF", "#ef6c00"])
    explainer = shap.KernelExplainer(model_wrapper, all_bulk_data)
    shap_values = explainer(all_bulk_data.values[0:,:])


    shap.summary_plot(
        shap_values[:, :, i],
        all_bulk_data,
        plot_type='dot',
        feature_names=all_bulk_data.columns,
        cmap=custom_cmap,
        show=False
    )

    plt.title(f'All Bulk Samples {titles[i]}', fontsize=16)
    plt.tight_layout()
    plt.show()
    plt.savefig(f'shap_analysis_{titles[i]}.pdf')

In [ ]:
# with open('bulk/kernel_shap_values.pkl', 'wb') as f:
#     pickle.dump(shap_values, f)

In [ ]:
shared_bulk_data_mask.shape, unshared_bulk_data.shape

In [ ]:
with open('./data/kernel_shap_values.pkl', 'wb') as f:
    pickle.dump(shap_values, f)

In [ ]:
with open('./data/explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)